In [3]:
import csv
import random
import math
import time


# ==========================================
# 1️⃣ Load Dataset
# ==========================================
FILE_PATH = "Mall_Customers.csv"
INCOME_COLUMN = "Annual Income (k$)"
SPENDING_COLUMN = "Spending Score (1-100)"

customer_data = []

with open(FILE_PATH, mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        income_value = float(row[INCOME_COLUMN])
        spending_value = float(row[SPENDING_COLUMN])
        customer_data.append([income_value, spending_value])


# ==========================================
# 2️⃣ Standardization
# ==========================================
def calculate_mean_and_std(values):
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    std_dev = math.sqrt(variance)
    return mean_value, std_dev


income_list = [point[0] for point in customer_data]
spending_list = [point[1] for point in customer_data]

income_mean, income_std = calculate_mean_and_std(income_list)
spending_mean, spending_std = calculate_mean_and_std(spending_list)

scaled_customers = [
    [
        (point[0] - income_mean) / income_std,
        (point[1] - spending_mean) / spending_std
    ]
    for point in customer_data
]


# ==========================================
# 3️⃣ K-MEANS FUNCTION
# ==========================================
def run_kmeans(data, k=5, max_iters=100000):
    centroids = random.sample(data, k)

    def dist(p, c):
        return (p[0] - c[0])**2 + (p[1] - c[1])**2

    for _ in range(max_iters):
        clusters = {i: [] for i in range(k)}

        for point in data:
            distances = [dist(point, c) for c in centroids]
            cluster_id = distances.index(min(distances))
            clusters[cluster_id].append(point)

        new_centroids = []
        for i in range(k):
            cluster_points = clusters[i]
            if cluster_points:
                mean_x = sum(p[0] for p in cluster_points) / len(cluster_points)
                mean_y = sum(p[1] for p in cluster_points) / len(cluster_points)
                new_centroids.append([mean_x, mean_y])
            else:
                new_centroids.append(random.choice(data))

        centroids = new_centroids

    return centroids


# ==========================================
# 4️⃣ K-MEDIANS FUNCTION
# ==========================================
def run_kmedians(data, k=5, max_iters=100000):
    centroids = random.sample(data, k)

    def dist(p, c):
        return abs(p[0] - c[0]) + abs(p[1] - c[1])

    for _ in range(max_iters):
        clusters = {i: [] for i in range(k)}

        for point in data:
            distances = [dist(point, c) for c in centroids]
            cluster_id = distances.index(min(distances))
            clusters[cluster_id].append(point)

        new_centroids = []
        for i in range(k):
            cluster_points = clusters[i]
            if cluster_points:
                sorted_x = sorted(p[0] for p in cluster_points)
                sorted_y = sorted(p[1] for p in cluster_points)

                mid = len(cluster_points) // 2
                if len(cluster_points) % 2 == 0:
                    median_x = (sorted_x[mid-1] + sorted_x[mid]) / 2
                    median_y = (sorted_y[mid-1] + sorted_y[mid]) / 2
                else:
                    median_x = sorted_x[mid]
                    median_y = sorted_y[mid]

                new_centroids.append([median_x, median_y])
            else:
                new_centroids.append(random.choice(data))

        centroids = new_centroids

    return centroids


# ==========================================
# 5️⃣ BENCHMARK
# ==========================================
random.seed(42)

start_time = time.perf_counter()
kmeans_centroids = run_kmeans(scaled_customers)
kmeans_time = time.perf_counter() - start_time

start_time = time.perf_counter()
kmedians_centroids = run_kmedians(scaled_customers)
kmedians_time = time.perf_counter() - start_time


# ==========================================
# 6️⃣ RESULTS
# ==========================================
print("\n===== Execution Time Comparison =====")
print(f"K-Means Time   : {kmeans_time:.6f} seconds")
print(f"K-Medians Time : {kmedians_time:.6f} seconds")

if kmeans_time < kmedians_time:
    print("K-Means is faster.")
else:
    print("K-Medians is faster.")

print("\nTime Difference:", abs(kmeans_time - kmedians_time))


===== Execution Time Comparison =====
K-Means Time   : 45.450455 seconds
K-Medians Time : 26.770847 seconds
K-Medians is faster.

Time Difference: 18.679608400008874
